In [1]:
import json
from pathlib import Path

root = Path('/data/train/train_60')
camconf_files = sorted(root.glob('*/camconf.json'))

if not camconf_files:
    raise FileNotFoundError(f'未找到 camconf.json: {root}/ */camconf.json')

x_vals, y_vals = [], []
invalid_files = []

def extract_xy(location):
    """兼容常见 location 格式并返回 (x, y)"""
    if isinstance(location, dict):
        x = location.get('x')
        y = location.get('y')
        # 兼容大写键
        if x is None:
            x = location.get('X')
        if y is None:
            y = location.get('Y')
        return x, y

    if isinstance(location, (list, tuple)) and len(location) >= 2:
        return location[0], location[1]

    return None, None

for fp in camconf_files:
    try:
        with fp.open('r', encoding='utf-8') as f:
            data = json.load(f)

        # 兼容几种可能字段路径
        location = data.get('location')
        if location is None and isinstance(data.get('camera'), dict):
            location = data['camera'].get('location')

        x, y = extract_xy(location)

        if x is None or y is None:
            invalid_files.append(str(fp))
            continue

        x_vals.append(float(x))
        y_vals.append(float(y))

    except Exception:
        invalid_files.append(str(fp))

if not x_vals or not y_vals:
    raise ValueError('没有成功提取到任何可用的 location.x / location.y')

result = {
    'x_min': min(x_vals),
    'x_max': max(x_vals),
    'y_min': min(y_vals),
    'y_max': max(y_vals),
    'total_camconf_files': len(camconf_files),
    'valid_files': len(x_vals),
    'invalid_files': len(invalid_files),
}

print('最终范围:')
print(f"x: [{result['x_min']}, {result['x_max']}]")
print(f"y: [{result['y_min']}, {result['y_max']}]")

# 如需查看异常文件可取消注释
# print('\n解析失败文件示例(前20个):')
# print('\n'.join(invalid_files[:20]))

最终范围:
x: [-1499.9154052734375, 99.99412536621094]
y: [-99.99513244628906, 399.9464416503906]


In [2]:
# 输出最终范围（若上一个单元已执行）
try:
    print(f"x: [{result['x_min']}, {result['x_max']}]")
    print(f"y: [{result['y_min']}, {result['y_max']}]")
except NameError:
    print('请先执行上一个单元。')

x: [-1499.9154052734375, 99.99412536621094]
y: [-99.99513244628906, 399.9464416503906]
